# Spatial GMM + Local Random Walker Hybrid

This notebook evaluates a new fourth model without changing `main.ipynb` or any existing model. It first runs the original single-slice XY Spatial GMM and its existing contour pipeline on slice 80. That mask is preserved as the base result. A Random Walker then runs only in small regions around the base mask and high-confidence posterior rescue seeds, using boundaries from the four MRI modalities.

Training volumes 1-250 fit the existing GMMs, volumes 251-300 select only the hybrid refinement parameters, and volumes 301-369 are used once for final evaluation. The original Spatial-GMM parameters are frozen and Z-axis context is not used. The original mask is always retained, so the hybrid cannot newly miss a tumor already detected by Spatial GMM.

Required packages: `scikit-image` and `optuna`. If needed, install them in the active environment with `python -m pip install --user scikit-image optuna` before running the imports.

In [8]:
!python -m pip install --user scikit-image optuna

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
You should consider upgrading via the '/bin/python -m pip install --upgrade pip' command.


In [9]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from config import PROJECT_ROOT
from gmm_random_walker.evaluation import evaluate_hybrid_test_set
from gmm_random_walker.model import evaluate_hybrid_volume
from gmm_random_walker.optimization import (
    HYBRID_OPTIMIZATION_VERSION,
    HYBRID_PARAMETER_NAMES,
    load_spatial_parameters,
    run_hybrid_optimization,
)

%matplotlib inline

dataset director:  /truenas/home/ratzabiy/MRI_2026_datasets/Brats/BraTS2020_training_data/content/data


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Validation-only parameter selection

The original central-slice Spatial-GMM parameters are loaded from `spatial_gmm_best_params.npz` and frozen. Optuna tunes only local Random-Walker rescue and refinement parameters. An exact no-op trial is always included. A hybrid candidate is accepted only if it does not lower validation Dice, add misses or tumor-free false positives, or reduce precision by more than 0.01 relative to the Spatial GMM.

In [10]:
PARAMETER_PATH = Path(PROJECT_ROOT) / "saved_parameters" / "gmm_random_walker_best_params.npz"
N_OPTIMIZATION_TRIALS = 30
REOPTIMIZE = False

def load_or_optimize_parameters():
    if PARAMETER_PATH.exists() and not REOPTIMIZE:
        with np.load(PARAMETER_PATH) as saved:
            missing = [name for name in HYBRID_PARAMETER_NAMES if name not in saved.files]
            version = int(saved["optimization_version"]) if "optimization_version" in saved.files else 0
            if not missing and version == HYBRID_OPTIMIZATION_VERSION:
                print(f"Loaded validation-selected parameters from {PARAMETER_PATH}")
                return {name: saved[name].item() for name in HYBRID_PARAMETER_NAMES}
    return run_hybrid_optimization(spatial_params, n_trials=N_OPTIMIZATION_TRIALS)

spatial_params = load_spatial_parameters()
hybrid_params = load_or_optimize_parameters()
print("Frozen one-slice Spatial-GMM parameters:")
display(pd.DataFrame(spatial_params.items(), columns=["Spatial parameter", "Value"]))
print("Validation-selected hybrid parameters:")
display(pd.DataFrame(hybrid_params.items(), columns=["Hybrid parameter", "Value"]))

Caching 50 validation slices...


[I 2026-08-12 12:48:10,207] A new study created in memory with name: no-name-23f86db3-6d6a-46f2-a5f8-bc61fb28d70e
  0%|          | 0/40 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/scipy/sparse/_index.py:146: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil_matrix is more efficient.
  self._set_arrayXarray(i, j, x)
/truenas/home/ratzabiy/.local/lib/python3.10/site-packages/skimage/_shared/utils.py:456: UserWarning: The probability range is outside [0, 1] given the tolerance `prob_tol`. Consider decreasing `beta` and/or decreasing `tol`.
  return func(*args, **kwargs)
Best trial: 0. Best value: 0.420798:   2%|▎         | 1/40 [00:34<22:36, 34.77s/it]

[I 2026-08-12 12:48:44,986] Trial 0 finished with value: 0.4207982132666286 and parameters: {'lambda_val': 0.15, 'tumor_prior_scale': 1.0, 'tumor_seed_threshold': 0.65, 'healthy_seed_threshold': 0.03, 'min_tumor_seed_pixels': 5, 'beta': 90.0, 'posterior_weight': 1.0}. Best is trial 0 with value: 0.4207982132666286.


Best trial: 1. Best value: 0.557267:   5%|▌         | 2/40 [01:11<22:39, 35.79s/it]

[I 2026-08-12 12:49:21,489] Trial 1 finished with value: 0.5572671334683006 and parameters: {'lambda_val': 0.2, 'tumor_prior_scale': 2.678154091306088, 'tumor_seed_threshold': 0.75, 'healthy_seed_threshold': 0.12, 'min_tumor_seed_pixels': 3, 'beta': 29.658005883239866, 'posterior_weight': 0.25}. Best is trial 1 with value: 0.5572671334683006.


...

Best trial: 13. Best value: 0.56934:  40%|████      | 16/40 [09:31<14:09, 35.39s/it]

[I 2026-08-12 12:57:42,112] Trial 15 finished with value: 0.4415855749590707 and parameters: {'lambda_val': 0.4, 'tumor_prior_scale': 1.0548032563983094, 'tumor_seed_threshold': 0.65, 'healthy_seed_threshold': 0.06999999999999999, 'min_tumor_seed_pixels': 10, 'beta': 23.375807974830405, 'posterior_weight': 0.5}. Best is trial 13 with value: 0.5693395285173215.


Best trial: 13. Best value: 0.56934:  40%|████      | 16/40 [09:49<14:43, 36.82s/it]


[W 2026-08-12 12:57:59,389] Trial 16 failed with parameters: {'lambda_val': 0.2, 'tumor_prior_scale': 1.9888234568804886, 'tumor_seed_threshold': 0.8, 'healthy_seed_threshold': 0.12, 'min_tumor_seed_pixels': 3, 'beta': 53.40078444896406, 'posterior_weight': 1.25} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/truenas/home/ratzabiy/.local/lib/python3.10/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/truenas/home/ratzabiy/Brain_Tumor_Segmentation/gmm_random_walker/optimization.py", line 124, in objective
    metrics = evaluate_parameters(cache, suggest_parameters(trial))
  File "/truenas/home/ratzabiy/Brain_Tumor_Segmentation/gmm_random_walker/optimization.py", line 71, in evaluate_parameters
    result = segment_with_random_walker(
  File "/truenas/home/ratzabiy/Brain_Tumor_Segmentation/gmm_random_walker/model.py", line 74, in segment_with_random_walker
    labels = random_wal

KeyboardInterrupt: 

## Final held-out test evaluation

Run this cell only after the validation parameters are frozen. Results are written to `output/gmm_random_walker/` and do not overwrite the existing evaluation files.

In [ ]:
RW_RESULTS = evaluate_hybrid_test_set(spatial_params, **hybrid_params)

## Comparison with saved existing-model results

The table reads the existing result files without rerunning or modifying those models.

In [ ]:
comparison_rows = []
existing_models = {
    "Baseline GMM": "baseline_gmm_metrics.npz",
    "Spatial GMM": "spatial_gmm_metrics.npz",
    "Spatial GMM + NDI": "spatial_gmm_ndi_metrics.npz",
}
for model_name, filename in existing_models.items():
    path = Path(PROJECT_ROOT) / "output" / "evaluation_scores" / filename
    if path.exists():
        with np.load(path) as result:
            comparison_rows.append({
                "Model": model_name,
                "Dice mean": result["mean_dice"].item(),
                "Tumor-present Dice": result["tumor_present_mean_dice"].item(),
                "Precision": result["tumor_present_mean_precision"].item(),
                "Recall": result["tumor_present_mean_recall"].item(),
                "IoU mean": result["mean_iou"].item(),
                "Missed tumors": f"{result['missed_tumors_count'].item()}/{result['gt_tumors_count'].item()}",
                "Empty-slice FP": result["tumor_free_false_positives"].item(),
            })
comparison_rows.append({
    "Model": "Spatial GMM + Local Random Walker",
    "Dice mean": RW_RESULTS["mean_dice"],
    "Tumor-present Dice": RW_RESULTS["tumor_present_mean_dice"],
    "Precision": RW_RESULTS["tumor_present_mean_precision"],
    "Recall": RW_RESULTS["tumor_present_mean_recall"],
    "IoU mean": RW_RESULTS["mean_iou"],
    "Missed tumors": f"{RW_RESULTS['missed_tumors_count']}/{RW_RESULTS['gt_tumors_count']}",
    "Empty-slice FP": RW_RESULTS["tumor_free_false_positives"],
})
comparison_table = pd.DataFrame(comparison_rows)
display(comparison_table.round(4))

## Missed tumors and qualitative inspection

In [ ]:
missed_volumes = list(map(int, RW_RESULTS["missed_volume_numbers"]))
print("Completely missed tumors:", missed_volumes)
inspection_volumes = missed_volumes[:6]
if inspection_volumes:
    fig, axes = plt.subplots(len(inspection_volumes), 6, figsize=(19, 3.2 * len(inspection_volumes)), squeeze=False)
    for row, volume in enumerate(inspection_volumes):
        details = evaluate_hybrid_volume(volume, spatial_params, return_details=True, **hybrid_params)
        panels = [
            (details["image"][:, :, 3], "FLAIR"),
            (details["posterior"], "GMM tumor posterior"),
            (details["base_prediction"], f"Spatial base (Dice={details['base_dice']:.3f})"),
            (details["rescue_tumor_seeds"], "Posterior rescue seeds"),
            (details["prediction"], f"Hybrid (Dice={details['dice']:.3f})"),
            (details["ground_truth"], "Ground truth"),
        ]
        for column, (panel, title) in enumerate(panels):
            axes[row, column].imshow(panel, cmap="gray")
            axes[row, column].set_title(f"Volume {volume}: {title}")
            axes[row, column].axis("off")
    plt.tight_layout()
    figure_path = Path(PROJECT_ROOT) / "output" / "gmm_random_walker" / "missed_tumor_inspection.png"
    plt.savefig(figure_path, dpi=220, bbox_inches="tight")
    plt.show()